# RNN v1 — Fall Detection (LSTM / GRU, ONNX / ST Edge AI 최적화)

ST Edge AI 지원 ops 목록 기준, LSTM/GRU 모두 `float32` 지원 확인됨.

| 모델 | 특징 |
|------|------|
| **LSTM** | 장기 의존성 포착 강함, 파라미터 많음 |
| **GRU** | LSTM 경량화 버전, 속도/메모리 우위 |

두 모델 모두 동일 학습 파이프라인 사용, `MODEL_TYPE`으로 전환 가능.

**사용 ONNX ops (ST Edge AI 지원 목록 기준)**  
`LSTM` / `GRU`, `Reshape`, `Transpose`, `Gemm`, `Relu`, `Dropout`

In [ ]:
# Colab 환경 설치
# !pip install onnx onnxruntime

## Cell 1. 모델 정의

In [ ]:
import os, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

# ── 하이퍼파라미터 ──────────────────────────────────────────────
WINDOW      = 60      # 15fps × 4sec
N_FEATURES  = 55      # 17 keypoints × 3 (y, x, score) = 51 + 4 engineered
N_CLASSES   = 2
HIDDEN_SIZE = 128
NUM_LAYERS  = 2
DROPOUT     = 0.3
BATCH_SIZE  = 128
EPOCHS      = 40
LR          = 1e-3
PATIENCE    = 8

# ── 모델 선택: 'LSTM' 또는 'GRU' ──────────────────────────────
MODEL_TYPE  = 'LSTM'

In [ ]:
class FallRNN(nn.Module):
    """
    LSTM 또는 GRU 기반 낙상 감지 모델.

    Input : (B, 60, 55)  — (Batch, Time, Features)  ← RNN 표준 입력 형태
    Output: (B, 2)       — (Batch, n_classes)

    ONNX ops:
      LSTM → LSTM, Reshape, Transpose, Gemm, Relu, Dropout
      GRU  → GRU,  Reshape, Transpose, Gemm, Relu, Dropout

    주의: bidirectional=True 사용 시 hidden 크기 2배 —
          ST Edge AI specific 제약 확인 권장.
          기본값 bidirectional=False (단방향)로 안전하게 설정.
    """
    def __init__(self, n_features=N_FEATURES, n_classes=N_CLASSES,
                 hidden_size=HIDDEN_SIZE, num_layers=NUM_LAYERS,
                 dropout=DROPOUT, model_type='LSTM', bidirectional=False):
        super().__init__()
        self.model_type    = model_type.upper()
        self.hidden_size   = hidden_size
        self.num_layers    = num_layers
        self.bidirectional = bidirectional
        directions         = 2 if bidirectional else 1

        rnn_cls = nn.LSTM if self.model_type == 'LSTM' else nn.GRU
        self.rnn = rnn_cls(
            input_size=n_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,       # (B, T, F) 입력
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=bidirectional,
        )

        # 마지막 타임스텝의 hidden → classifier
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size * directions, n_classes)

    def forward(self, x):
        # x: (B, T, F) = (B, 60, 55)
        out, _ = self.rnn(x)          # out: (B, T, hidden*directions)
        last   = out[:, -1, :]        # 마지막 타임스텝: (B, hidden*directions)
        last   = self.dropout(last)
        return self.classifier(last)  # (B, 2)


model = FallRNN(model_type=MODEL_TYPE).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'{MODEL_TYPE} v1 파라미터 수: {n_params:,}')
print(model)

## Cell 2. 데이터 로딩 & Dataset

In [ ]:
def load_and_prepare_data(file_path, usage_ratio=0.95):
    df = pd.read_csv(file_path)
    df = df.sort_values(['video_id', 'frame']).reset_index(drop=True)

    feature_cols = [c for c in df.columns
                    if c not in ('video_id', 'frame', 'label')]

    all_vids   = df['video_id'].unique()
    n_use      = int(len(all_vids) * usage_ratio)
    train_vids = all_vids[:n_use]
    test_vids  = all_vids[n_use:]

    df_use  = df[df['video_id'].isin(train_vids)].reset_index(drop=True)
    df_test = df[df['video_id'].isin(test_vids)].reset_index(drop=True)

    print(f'학습용 비디오: {len(train_vids)} | 테스트용: {len(test_vids)}')
    print(f'피처 수: {len(feature_cols)}')
    return df_use, df_test, feature_cols


class FallDataset(Dataset):
    """
    video_id 단위 슬라이딩 윈도우.
    RNN 입력 형태: (Time, Features) = (60, 55)  ← TCN과 반대
    """
    def __init__(self, df, feature_cols, window_size=WINDOW, mode='train'):
        self.samples = []

        for vid, grp in df.groupby('video_id'):
            grp = grp.reset_index(drop=True)
            X   = grp[feature_cols].values.astype(np.float32)  # (T, 55)
            y   = grp['label'].values
            T   = len(grp)

            if T < window_size:
                continue

            for start in range(T - window_size + 1):
                win_y   = int(y[start + window_size - 1])
                is_fall = (win_y == 1)

                if mode == 'train':
                    stride = 1 if is_fall else 5
                    if start % stride != 0:
                        continue

                x_win = X[start:start + window_size]   # (60, 55)
                self.samples.append((x_win, win_y))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        x, y = self.samples[idx]
        # RNN 입력 형태: (Time, Features) = (60, 55)  — TCN과 달리 Transpose 불필요
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.long)


# ── 실행 ────────────────────────────────────────────────────────
DATA_PATH = 'final_dataset.csv'

df_use, df_test, feature_cols = load_and_prepare_data(DATA_PATH)

videos = df_use['video_id'].unique()
gss    = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
tr_idx, va_idx = next(gss.split(videos, groups=videos))

df_train = df_use[df_use['video_id'].isin(videos[tr_idx])]
df_val   = df_use[df_use['video_id'].isin(videos[va_idx])]

train_ds = FallDataset(df_train, feature_cols, mode='train')
val_ds   = FallDataset(df_val,   feature_cols, mode='val')

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train: {len(train_ds):,}  Val: {len(val_ds):,}')

labels   = [s[1] for s in train_ds.samples]
n_normal, n_fall = labels.count(0), labels.count(1)
class_weights = torch.tensor([1.0, n_normal / n_fall], dtype=torch.float32).to(DEVICE)
print(f'정상: {n_normal:,}  낙상: {n_fall:,}  fall weight: {class_weights[1]:.2f}')

## Cell 3. 학습

In [ ]:
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=3, verbose=True)


def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    ctx = torch.enable_grad() if train else torch.no_grad()

    with ctx:
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits = model(x)
            loss   = criterion(logits, y)

            if train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            total_loss += loss.item() * len(y)
            correct    += (logits.argmax(1) == y).sum().item()
            total      += len(y)

    return total_loss / total, correct / total


MODEL_SAVE = f'fall_detection_rnn_v1_{MODEL_TYPE.lower()}.pth'
history    = {'train_loss': [], 'val_loss': [], 'val_acc': []}
best_val_acc, patience_cnt = 0.0, 0

for epoch in range(1, EPOCHS + 1):
    tr_loss, _        = run_epoch(train_loader, train=True)
    val_loss, val_acc = run_epoch(val_loader,   train=False)
    scheduler.step(val_acc)

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    mark = ''
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), MODEL_SAVE)
        patience_cnt = 0
        mark = ' ◀ saved'
    else:
        patience_cnt += 1

    print(f'Epoch {epoch:02d}/{EPOCHS}  '
          f'Loss {tr_loss:.4f}  Val Loss {val_loss:.4f}  Val Acc {val_acc:.4f}{mark}')

    if patience_cnt >= PATIENCE:
        print(f'Early stopping at epoch {epoch}')
        break

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history['train_loss'], label='Train')
ax1.plot(history['val_loss'],   label='Val')
ax1.set_title('Loss'); ax1.legend()
ax2.plot(history['val_acc'])
ax2.set_title('Val Accuracy')
plt.tight_layout(); plt.show()
print(f'최고 Val Accuracy: {best_val_acc:.4f}')

## Cell 4. 테스트셋 평가

In [ ]:
model.load_state_dict(torch.load(MODEL_SAVE))
model.eval()

test_ds     = FallDataset(df_test, feature_cols, mode='test')
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

all_preds, all_labels = [], []
with torch.no_grad():
    for x, y in test_loader:
        logits = model(x.to(DEVICE))
        all_preds.extend(logits.argmax(1).cpu().tolist())
        all_labels.extend(y.tolist())

print(f'=== Test Set 결과 ({MODEL_TYPE}) ===')
print(classification_report(all_labels, all_preds, target_names=['Normal', 'Fall']))
print('Confusion Matrix:')
print(confusion_matrix(all_labels, all_preds))

## Cell 5. ONNX Export & 검증

> **LSTM/GRU ONNX export 주의사항**  
> PyTorch의 `nn.LSTM`은 export 시 ONNX `LSTM` op으로 변환됨.  
> ST Edge AI 지원 목록에 `LSTM`, `GRU` 모두 `float32` 확인됨 (specific 조건 있음).  
> `bidirectional=True` 사용 시 specific 조건 재확인 필요.

In [ ]:
import onnx
import onnxruntime as ort

model.load_state_dict(torch.load(MODEL_SAVE))
model.eval().cpu()

ONNX_PATH   = f'fall_detection_rnn_v1_{MODEL_TYPE.lower()}.onnx'
dummy_input = torch.randn(1, WINDOW, N_FEATURES)  # (1, 60, 55) — RNN 입력 형태

torch.onnx.export(
    model,
    dummy_input,
    ONNX_PATH,
    opset_version=13,
    input_names=['input'],    # shape (1, 60, 55)
    output_names=['output'],  # shape (1, 2)
    dynamic_axes={
        'input':  {0: 'batch'},
        'output': {0: 'batch'},
    },
)
print(f'ONNX 저장: {ONNX_PATH}')

# ── 유효성 검사 ──────────────────────────────────────────────
onnx_model = onnx.load(ONNX_PATH)
onnx.checker.check_model(onnx_model)
print('ONNX checker 통과')

with torch.no_grad():
    pt_out = model(dummy_input).numpy()

sess    = ort.InferenceSession(ONNX_PATH)
ort_out = sess.run(None, {'input': dummy_input.numpy()})[0]

max_diff = float(np.abs(pt_out - ort_out).max())
print(f'PyTorch vs ONNX 최대 오차: {max_diff:.2e}')
assert max_diff < 1e-4, '출력 불일치 — ONNX export 재확인 필요'
print('검증 완료 → ST Edge AI X-CUBE-AI에 import 가능')

used_ops = sorted({n.op_type for n in onnx_model.graph.node})
print(f'\n사용 ops ({len(used_ops)}개): {used_ops}')

## (선택) GRU로 전환하여 재학습

위 Cell 1에서 `MODEL_TYPE = 'GRU'`로 바꾼 뒤 전체 재실행.  
GRU는 LSTM 대비 파라미터 약 25% 적고, 동등하거나 더 나은 성능을 내는 경우 많음.